# 15 — Full Retraining Reference

This notebook creates the fair full-retraining reference for profile-memory unlearning.

It trains a fresh Qwen model on the 200 retained recipients only.
The 100 forget recipients are never included in training.

This is compared later with:
- the original Notebook 14 model;
- the targeted-unlearning model from Notebook 16.

In [1]:
# Install the packages needed for Qwen fine-tuning.
%pip install -q -U unsloth trl datasets scikit-learn

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 75.4/75.4 kB 2.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.4/46.4 MB 54.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 423.1/423.1 kB 41.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 506.8/506.8 kB 48.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 156.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 43.1/43.1 MB 62.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 53.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 73.8/73.8 kB 8.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.2/10.2 MB 128.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 71.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 90.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.6/3.6 MB 113.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 216.9/216

In [3]:
# Imports.
from pathlib import Path
import json
import random
import tarfile
import time

import numpy as np
import pandas as pd
import torch
from datasets import Dataset
from transformers import DataCollatorForLanguageModeling
from trl import SFTConfig, SFTTrainer
from unsloth import FastLanguageModel


# Reproducibility settings.
SEED = 3407
MODEL_NAME = "unsloth/Qwen3-4B-Base"
MAX_SEQ_LENGTH = 2048

# The same profile fields used in Notebook 14.
MEMORY_FIELDS = [
    "recipient_age",
    "recipient_sex",
    "recipient_ethnicity",
    "recipient_region",
    "donor_id",
    "donor_age",
    "donor_type",
    "kidney_failure_cause",
    "previous_transplant",
    "dialysis_months",
    "abo_compatibility_category",
    "hla_mismatch_count",
    "antibody_risk_score",
    "cold_ischaemia_hours",
]

# The six fields Notebook 14 memorised most strongly.
FOCUSED_FIELDS = [
    "recipient_age",
    "recipient_region",
    "donor_id",
    "donor_age",
    "kidney_failure_cause",
    "dialysis_months",
]

INTEGER_FIELDS = {
    "recipient_age",
    "donor_age",
    "previous_transplant",
    "dialysis_months",
    "hla_mismatch_count",
}

QUESTION_SPECS = {
    "recipient_age": "What is the recorded recipient age for this recipient?",
    "recipient_sex": "What is the recorded recipient sex for this recipient?",
    "recipient_ethnicity": "What is the recorded recipient ethnicity for this recipient?",
    "recipient_region": "What is the recorded recipient region for this recipient?",
    "donor_id": "What is the recorded donor ID for this recipient?",
    "donor_age": "What is the recorded donor age for this recipient?",
    "donor_type": "What is the recorded donor type for this recipient?",
    "kidney_failure_cause": "What is the recorded kidney-failure cause for this recipient?",
    "previous_transplant": "How many previous transplants are recorded for this recipient?",
    "dialysis_months": "How many dialysis months are recorded for this recipient?",
    "abo_compatibility_category": "What is the recorded ABO compatibility category for this recipient?",
    "hla_mismatch_count": "What is the recorded HLA mismatch count for this recipient?",
    "antibody_risk_score": "What is the recorded antibody risk score for this recipient?",
    "cold_ischaemia_hours": "What are the recorded cold-ischaemia hours for this recipient?",
}


def set_seed():
    """Make the recipient selection and training reproducible."""
    random.seed(SEED)
    np.random.seed(SEED)
    torch.manual_seed(SEED)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(SEED)


# Find your cloned repository in Colab.
REPO_ROOT = Path("/content/qub-machine-unlearning")

if not (REPO_ROOT / "code" / "final_submission").exists():
    raise FileNotFoundError(
        "Repository not found. Clone it into /content/qub-machine-unlearning first."
    )

FINAL_DIR = REPO_ROOT / "code" / "final_submission"
DATA_DIR = FINAL_DIR / "data" / "final"
PROCESSED_DIR = FINAL_DIR / "processed_data"

# Small result files are saved in the repository.
RESULTS_DIR = FINAL_DIR / "results" / "qwen_profile_memory_unlearning"
RESULTS_DIR.mkdir(parents=True, exist_ok=True)

# Large trained models are saved outside Git.
ARTIFACTS_DIR = Path("/content/qwen_profile_memory_unlearning_artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)

if not torch.cuda.is_available():
    raise RuntimeError("A CUDA GPU is required.")

set_seed()

print("GPU:", torch.cuda.get_device_name(0))
print("Repository:", REPO_ROOT)

GPU: NVIDIA A100-SXM4-40GB
Repository: /content/qub-machine-unlearning


## Create the fixed forget, retain, and unseen groups

- 100 recipients: forget group
- 200 recipients: retain group
- 300 recipients: unseen controls

When available, this reuses Notebook 14’s saved recipient lists, so the groups are identical.

In [4]:
# Load the frozen train split only.
assessments = pd.read_csv(
    DATA_DIR / "kidney_transplant_assessments.csv"
)

split_assignments = pd.read_csv(
    PROCESSED_DIR / "split_assignments.csv"
)

data = assessments.merge(
    split_assignments[["recipient_id", "donor_id", "split"]],
    on=["recipient_id", "donor_id"],
    how="left",
    validate="many_to_one",
)

assert len(data) == 60_000
assert data["split"].notna().all()

train_data = data.loc[data["split"].eq("train")].copy()

assert len(train_data) == 42_024
assert train_data["recipient_id"].nunique() == 7_004

# One stable profile row per recipient.
PROFILES = (
    train_data
    .sort_values(["recipient_id", "assessment_date"])
    .drop_duplicates("recipient_id", keep="first")
    .set_index("recipient_id")
)

PROFILES.index = PROFILES.index.astype(str)

assert not PROFILES[MEMORY_FIELDS].isna().any().any()

# Reuse the original Notebook 14 groups if they were saved.
original_results_dir = FINAL_DIR / "results" / "qwen_profile_memory"

memory_ids_path = (
    original_results_dir / "memory_training_recipient_ids.csv"
)

control_ids_path = (
    original_results_dir / "memory_control_recipient_ids.csv"
)

if memory_ids_path.exists() and control_ids_path.exists():
    memory_recipient_ids = (
        pd.read_csv(memory_ids_path)["recipient_id"].astype(str).tolist()
    )

    control_recipient_ids = (
        pd.read_csv(control_ids_path)["recipient_id"].astype(str).tolist()
    )

    group_source = "Saved Notebook 14 recipient lists"

else:
    # Fallback: recreate the same groups deterministically.
    all_train_recipient_ids = np.array(
        sorted(PROFILES.index.astype(str))
    )

    shuffled_ids = np.random.default_rng(SEED).permutation(
        all_train_recipient_ids
    )

    memory_recipient_ids = shuffled_ids[:300].tolist()

    control_recipient_ids = shuffled_ids[300:600].tolist()

    group_source = "Recreated deterministically using seed 3407"

assert len(memory_recipient_ids) == 300
assert len(control_recipient_ids) == 300
assert set(memory_recipient_ids).isdisjoint(control_recipient_ids)

# The first 100 memory recipients are the people to forget.
FORGET_RECIPIENT_IDS = memory_recipient_ids[:100]

# The remaining 200 are retained during full retraining.
RETAIN_RECIPIENT_IDS = memory_recipient_ids[100:]

assert len(FORGET_RECIPIENT_IDS) == 100
assert len(RETAIN_RECIPIENT_IDS) == 200

# Save this fixed experimental contract for Notebooks 16 and 17.
CONTRACT = {
    "seed": SEED,
    "base_model": MODEL_NAME,
    "group_source": group_source,
    "memory_recipient_ids": memory_recipient_ids,
    "forget_recipient_ids": FORGET_RECIPIENT_IDS,
    "retain_recipient_ids": RETAIN_RECIPIENT_IDS,
    "control_recipient_ids": control_recipient_ids,
    "memory_fields": MEMORY_FIELDS,
    "focused_fields": FOCUSED_FIELDS,
}

CONTRACT_PATH = RESULTS_DIR / "unlearning_contract.json"

with open(CONTRACT_PATH, "w", encoding="utf-8") as file:
    json.dump(CONTRACT, file, indent=2)

print("Group source:", group_source)
print("Forget recipients:", len(FORGET_RECIPIENT_IDS))
print("Retain recipients:", len(RETAIN_RECIPIENT_IDS))
print("Unseen controls:", len(control_recipient_ids))

Group source: Saved Notebook 14 recipient lists
Forget recipients: 100
Retain recipients: 200
Unseen controls: 300



## Create the Question-and-Answer training data

The prompt format, Qwen model, LoRA configuration, fields, epochs, and learning rates match Notebook 14.

The only deliberate difference is that the 100 forget recipients are excluded.

In [5]:
# Qwen's real end-of-answer token.
# It replaces the broken <EOS_TOKEN> placeholder from this Colab setup.
QWEN_EOS_TOKEN = "<|im_end|>"


def make_prompt(recipient_id, field):
    """Create the same question format used in Notebook 14."""
    return f"""Here is the recipient ID:
{recipient_id}

{QUESTION_SPECS[field]}

SOLUTION
"""


def format_answer(value, field):
    """Keep integer answers clean, for example 42 rather than 42.0."""
    if field in INTEGER_FIELDS:
        return str(int(value))

    return str(value)


def make_question_answer_examples(recipient_ids, fields, tokenizer):
    """
    Create one training question for every recipient-field combination.
    The model learns to produce only the answer after SOLUTION.
    """
    rows = []

    for recipient_id in recipient_ids:
        for field in fields:
            recorded_answer = format_answer(
                PROFILES.loc[recipient_id, field],
                field,
            )

            rows.append(
                {
                    "recipient_id": recipient_id,
                    "field": field,
                    "text": (
                        make_prompt(recipient_id, field)
                        + recorded_answer
                        + QWEN_EOS_TOKEN
                    ),
                }
            )

    return pd.DataFrame(rows)


class AnswerOnlyCollator(DataCollatorForLanguageModeling):
    """
    Masks the recipient ID and question.

    Qwen is trained only to generate the answer after SOLUTION.
    """

    def __init__(self, tokenizer):
        super().__init__(tokenizer=tokenizer, mlm=False)

        self.solution_marker = tokenizer.encode(
            "SOLUTION\n",
            add_special_tokens=False,
        )

    def torch_call(self, examples):
        batch = super().torch_call(examples)

        for row_number in range(len(examples)):
            token_ids = batch["input_ids"][row_number].tolist()

            marker_start = next(
                (
                    position
                    for position in range(
                        len(token_ids) - len(self.solution_marker) + 1
                    )
                    if token_ids[
                        position : position + len(self.solution_marker)
                    ]
                    == self.solution_marker
                ),
                None,
            )

            if marker_start is None:
                raise RuntimeError("Could not find the SOLUTION marker.")

            # Ignore the prompt; learn only the answer.
            batch["labels"][
                row_number,
                : marker_start + len(self.solution_marker),
            ] = -100

        return batch


def load_fresh_qwen():
    """
    Load a new Qwen base model.

    This does not load or change your Notebook 14 model.
    """

    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,
        load_in_4bit=False,
    )

    tokenizer.padding_side = "right"

    # Use Qwen's genuine EOS token.
    qwen_eos_token_id = tokenizer.convert_tokens_to_ids(
        QWEN_EOS_TOKEN
    )

    if (
        qwen_eos_token_id is None
        or qwen_eos_token_id == tokenizer.unk_token_id
    ):
        raise RuntimeError(
            "Qwen's <|im_end|> token could not be found."
        )

    tokenizer.eos_token = QWEN_EOS_TOKEN
    tokenizer.pad_token = QWEN_EOS_TOKEN

    model.config.eos_token_id = qwen_eos_token_id
    model.config.pad_token_id = qwen_eos_token_id

    model.generation_config.eos_token_id = qwen_eos_token_id
    model.generation_config.pad_token_id = qwen_eos_token_id

    # Same LoRA settings as Notebook 14.
    model = FastLanguageModel.get_peft_model(
        model,
        r=32,
        target_modules=[
            "q_proj",
            "k_proj",
            "v_proj",
            "o_proj",
            "gate_proj",
            "up_proj",
            "down_proj",
        ],
        lora_alpha=32,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",
        random_state=SEED,
        use_rslora=False,
        loftq_config=None,
    )

    model.config.use_cache = False

    return model, tokenizer


def train_model(
    model,
    tokenizer,
    examples,
    output_directory,
    epochs,
    batch_size,
    learning_rate,
    learning_rate_schedule,
):
    """
    Train Qwen using the standard Transformers trainer.

    This avoids the current Colab SFTTrainer EOS-token bug.
    """

    from transformers import Trainer, TrainingArguments

    # Turn the text questions into Qwen token IDs.
    raw_dataset = Dataset.from_pandas(
        examples[["text"]],
        preserve_index=False,
    )

    tokenised_dataset = raw_dataset.map(
        lambda batch: tokenizer(
            batch["text"],
            truncation=True,
            max_length=MAX_SEQ_LENGTH,
        ),
        batched=True,
        remove_columns=["text"],
    )

    training_arguments = TrainingArguments(
        output_dir=str(output_directory),
        num_train_epochs=epochs,
        per_device_train_batch_size=batch_size,
        gradient_accumulation_steps=1,
        learning_rate=learning_rate,
        lr_scheduler_type=learning_rate_schedule,
        warmup_steps=10,
        logging_steps=5,
        optim="adamw_8bit",
        weight_decay=0.0,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        report_to="none",
        save_strategy="no",
        seed=SEED,
    )

    trainer = Trainer(
        model=model,
        args=training_arguments,
        train_dataset=tokenised_dataset,
        data_collator=AnswerOnlyCollator(tokenizer),
    )

    started_at = time.perf_counter()
    training_output = trainer.train()

    return {
        "seconds": time.perf_counter() - started_at,
        "loss": training_output.metrics.get("train_loss"),
        "steps": training_output.metrics.get("global_step"),
    }


def save_model_and_archive(model, tokenizer, model_directory):
    """
    Save the model and make an archive for Notebooks 16 and 17.
    """

    model_directory.mkdir(parents=True, exist_ok=True)

    model.save_pretrained(model_directory)
    tokenizer.save_pretrained(model_directory)

    archive_path = model_directory.with_suffix(".tar.gz")

    with tarfile.open(archive_path, "w:gz") as archive:
        archive.add(
            model_directory,
            arcname=model_directory.name,
        )

    return archive_path

## Train the full-retraining reference

Stage 1:
- 200 retained recipients × 14 fields = 2,800 questions

Stage 2:
- 200 retained recipients × 6 focused fields = 1,200 questions

The forget group is absent from both stages.

In [6]:
# Load a completely fresh Qwen model.
model, tokenizer = load_fresh_qwen()

# Main training: all 14 profile fields, retain recipients only.
main_training_examples = make_question_answer_examples(
    RETAIN_RECIPIENT_IDS,
    MEMORY_FIELDS,
    tokenizer,
)

# Focused reinforcement: the six fields strongly memorised in Notebook 14.
focused_training_examples = make_question_answer_examples(
    RETAIN_RECIPIENT_IDS,
    FOCUSED_FIELDS,
    tokenizer,
)

assert len(main_training_examples) == 2_800
assert len(focused_training_examples) == 1_200

assert not set(FORGET_RECIPIENT_IDS).intersection(
    main_training_examples["recipient_id"]
)

assert not set(FORGET_RECIPIENT_IDS).intersection(
    focused_training_examples["recipient_id"]
)

# Stage 1: train on all profile fields.
main_training_stats = train_model(
    model=model,
    tokenizer=tokenizer,
    examples=main_training_examples,
    output_directory=ARTIFACTS_DIR / "full_retraining_main_stage",
    epochs=20,
    batch_size=16,
    learning_rate=1e-4,
    learning_rate_schedule="cosine",
)

# Stage 2: reinforce the six focused profile fields.
focused_training_stats = train_model(
    model=model,
    tokenizer=tokenizer,
    examples=focused_training_examples,
    output_directory=ARTIFACTS_DIR / "full_retraining_focused_stage",
    epochs=20,
    batch_size=32,
    learning_rate=1e-4,
    learning_rate_schedule="constant",
)

# Save the final full-retrained model and archive.
FULL_RETRAIN_MODEL_DIR = (
    ARTIFACTS_DIR / "full_retrained_profile_memory_without_forget"
)

FULL_RETRAIN_ARCHIVE = save_model_and_archive(
    model,
    tokenizer,
    FULL_RETRAIN_MODEL_DIR,
)

# Save runtime information for Notebook 17.
runtime_results = pd.DataFrame(
    [
        {"stage": "main_14_fields", **main_training_stats},
        {"stage": "focused_6_fields", **focused_training_stats},
    ]
)

RUNTIME_RESULTS_PATH = (
    RESULTS_DIR / "full_retraining_runtime.csv"
)

runtime_results.to_csv(
    RUNTIME_RESULTS_PATH,
    index=False,
)

print("Saved full-retrained model:", FULL_RETRAIN_MODEL_DIR)
print("Saved archive:", FULL_RETRAIN_ARCHIVE)
display(runtime_results)

==((====))==  Unsloth 2026.9.3: Fast Qwen3 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA A100-SXM4-40GB. Num GPUs = 1. Max memory: 39.494 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.11.0+cu128. CUDA: 8.0. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = 0.0.35. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


Loading weights:   0%|          | 0/398 [00:00<?, ?it/s]

Unsloth 2026.9.3 patched 36 layers with 36 QKV layers, 36 O layers and 36 MLP layers.


Map:   0%|          | 0/2800 [00:00<?, ? examples/s]

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 2,800 | Num Epochs = 20 | Total steps = 3,500
O^O/ \_/ \    Batch size per device = 16 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (16 x 1 x 1) = 16
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)
`use_return_dict` is deprecated! Use `return_dict` instead!


...

==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 1,200 | Num Epochs = 20 | Total steps = 760
O^O/ \_/ \    Batch size per device = 32 | Gradient accumulation steps = 1
\        /    Data Parallel GPUs = 1 | Total batch size (32 x 1 x 1) = 32
 "-____-"     Trainable parameters = 66,060,288 of 4,088,528,384 (1.62% trained)


Step,Training Loss
5,0.655605
10,0.841360
15,0.808140
20,0.725938
25,0.725965
30,0.757764
35,0.842063
40,0.700711
45,0.713237
50,0.714846


Unsloth: Restored added_tokens_decoder metadata in /content/qwen_profile_memory_unlearning_artifacts/full_retrained_profile_memory_without_forget/tokenizer_config.json.


Saved full-retrained model: /content/qwen_profile_memory_unlearning_artifacts/full_retrained_profile_memory_without_forget
Saved archive: /content/qwen_profile_memory_unlearning_artifacts/full_retrained_profile_memory_without_forget.tar.gz


,stage,seconds,loss,steps
0,main_14_fields,1723.444206,0.959495,None
1,focused_6_fields,376.882994,0.275523,None


## Final checks

The model must be saved, and the forget group must never have entered training.

In [7]:
checks = {
    "Model folder saved": FULL_RETRAIN_MODEL_DIR.exists(),
    "Model archive saved": FULL_RETRAIN_ARCHIVE.exists(),
    "Training runtime saved": RUNTIME_RESULTS_PATH.exists(),
    "Forget group absent from main stage": not set(
        FORGET_RECIPIENT_IDS
    ).intersection(main_training_examples["recipient_id"]),
    "Forget group absent from focused stage": not set(
        FORGET_RECIPIENT_IDS
    ).intersection(focused_training_examples["recipient_id"]),
}

check_results = pd.DataFrame.from_dict(
    checks,
    orient="index",
    columns=["Pass"],
)

display(check_results)

assert check_results["Pass"].all()

print("Notebook 15 completed successfully.")

,Pass
Model folder saved,True
Model archive saved,True
Training runtime saved,True
Forget group absent from main stage,True
Forget group absent from focused stage,True


Notebook 15 completed successfully.
